# PrestaMatch: red neuronal (MLP)

Mismo procedimiento que el TP (variables, train / validation / test, ganancia esperada y presupuesto), usando una red neuronal con Keras.

## Librerías

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from tensorflow import keras

2026-10-19 22:08:35.418666: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-19 22:08:35.682281: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 22:08:35.965070: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-19 22:08:35.969695: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-19 22:08:38.333189: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

## Cargar los datos

In [2]:
train = pd.read_csv("ucse-ai-2026/solicitudes_train.csv")
bank = pd.read_csv("ucse-ai-2026/comportamiento_bancario.csv")
test = pd.read_csv("ucse-ai-2026/solicitudes_test.csv")

## Mismas variables que la otra solución

In [3]:
atrasos = bank.groupby("ID_Usuario")["Dias_Atraso_Otras_Deudas"]

max_atraso = atrasos.max().rename("max_dias_atraso")
promedio_atraso = atrasos.mean().rename("promedio_dias_atraso")

train_model = train.merge(max_atraso, on="ID_Usuario", how="left")
train_model = train_model.merge(promedio_atraso, on="ID_Usuario", how="left")

test_model = test.merge(max_atraso, on="ID_Usuario", how="left")
test_model = test_model.merge(promedio_atraso, on="ID_Usuario", how="left")

variables = [
    "max_dias_atraso",
    "promedio_dias_atraso",
    "Monto_Solicitado",
    "Tasa_Interes"
]

X = train_model[variables]
y = train_model["Estado_Final"]
X_test = test_model[variables]

## Train / validation y normalización de las entradas

In [4]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# El scaler aprende solo con train y se aplica igual a validation y test
scaler = StandardScaler()
X_tr_n = scaler.fit_transform(X_tr)
X_val_n = scaler.transform(X_val)
X_test_n = scaler.transform(X_test)

## Crear y entrenar la red neuronal (MLP)

In [5]:
keras.utils.set_random_seed(42)

red = keras.models.Sequential([
    keras.layers.Dense(16, activation="relu", input_shape=(len(variables),)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

red.compile(optimizer="adam", loss="binary_crossentropy")

red.fit(
    X_tr_n,
    y_tr,
    validation_data=(X_val_n, y_val),
    epochs=30,
    batch_size=64
)

Epoch 1/30
355/355 [==============================] - 2s 4ms/step - loss: 0.4285 - val_loss: 0.3447
Epoch 2/30
355/355 [==============================] - 1s 3ms/step - loss: 0.3474 - val_loss: 0.3373
Epoch 3/30
355/355 [==============================] - 1s 3ms/step - loss: 0.3434 - val_loss: 0.3343
Epoch 4/30
355/355 [==============================] - 1s 3ms/step - loss: 0.3398 - val_loss: 0.3342
Epoch 5/30
355/355 [==============================] - 1s 4ms/step - loss: 0.3388 - val_loss: 0.3322
Epoch 6/30
355/355 [==============================] - 1s 2ms/step - loss: 0.3373 - val_loss: 0.3320
Epoch 7/30
355/355 [==============================] - 1s 2ms/step - loss: 0.3354 - val_loss: 0.3315
Epoch 8/30
355/355 [==============================] - 1s 2ms/step - loss: 0.3342 - val_loss: 0.3319
Epoch 9/30
355/355 [==============================] - 1s 3ms/step - loss: 0.3338 - val_loss: 0.3321
Epoch 10/30
355/355 [==============================] - 1s 3ms/step - loss: 0.3321 - val_loss: 0.3309

## Función para calcular ganancia esperada y aprobar préstamos

In [6]:
def aprobar_prestamos(resultados, presupuesto=2_000_000):

    resultados["ganancia_esperada"] = (
        resultados["prob_pagado"]
        * resultados["Monto_Solicitado"]
        * resultados["Tasa_Interes"]
        - (1 - resultados["prob_pagado"])
        * resultados["Monto_Solicitado"]
    )

    resultados["ganancia_por_dolar"] = (
        resultados["ganancia_esperada"]
        / resultados["Monto_Solicitado"]
    )

    resultados = resultados.sort_values(
        "ganancia_por_dolar",
        ascending=False
    )

    monto_acumulado = 0
    resultados["aprobado"] = 0

    for idx, fila in resultados.iterrows():

        monto = fila["Monto_Solicitado"]

        if monto_acumulado + monto <= presupuesto:
            resultados.loc[idx, "aprobado"] = 1
            monto_acumulado += monto

    return resultados, monto_acumulado

## Evaluar la red en train y validation

In [7]:
prob_train = red.predict(X_tr_n).ravel()
prob_val = red.predict(X_val_n).ravel()

print("\nTRAIN:")
print(f"Accuracy: {accuracy_score(y_tr, (prob_train >= 0.5).astype(int)):.4f}")
print(f"Log Loss: {log_loss(y_tr, prob_train):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_tr, prob_train):.4f}")

print("\nVALIDATION:")
print(f"Accuracy: {accuracy_score(y_val, (prob_val >= 0.5).astype(int)):.4f}")
print(f"Log Loss: {log_loss(y_val, prob_val):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_val, prob_val):.4f}")

# Ganancia económica en validation
resultados_val = train_model.loc[X_val.index].copy()
resultados_val["prob_pagado"] = prob_val

resultados_val, _ = aprobar_prestamos(resultados_val)

resultados_val["ganancia_real"] = 0.0

mask_pago = (
    (resultados_val["aprobado"] == 1) &
    (resultados_val["Estado_Final"] == 1)
)
resultados_val.loc[mask_pago, "ganancia_real"] = (
    resultados_val.loc[mask_pago, "Monto_Solicitado"]
    * resultados_val.loc[mask_pago, "Tasa_Interes"]
)

mask_default = (
    (resultados_val["aprobado"] == 1) &
    (resultados_val["Estado_Final"] == 0)
)
resultados_val.loc[mask_default, "ganancia_real"] = (
    -resultados_val.loc[mask_default, "Monto_Solicitado"]
)

print(f"\nGanancia neta en validation: ${resultados_val['ganancia_real'].sum():,.2f}")

304/304 [==============================] - 0s 1ms/step

TRAIN:
Accuracy: 0.8619
Log Loss: 0.3275
ROC-AUC:  0.8390

VALIDATION:
Accuracy: 0.8606
Log Loss: 0.3314
ROC-AUC:  0.8334

Ganancia neta en validation: $469,880.86


## Predecir el test y armar la entrega

In [8]:
resultados = test_model.copy()
resultados["prob_pagado"] = red.predict(X_test_n).ravel()

resultados, monto_acumulado = aprobar_prestamos(resultados)

print("Monto total aprobado:", monto_acumulado)
print("Cantidad de préstamos aprobados:", resultados["aprobado"].sum())

entrega = resultados[[
    "ID_Solicitud",
    "aprobado"
]].rename(columns={
    "aprobado": "Decision_Aprobacion"
})

entrega.to_csv(
    "submission_red_neuronal.csv",
    index=False
)

96/96 [==============================] - 0s 1ms/step
Monto total aprobado: 1999600
Cantidad de préstamos aprobados: 765
